<!--nav--> [🗺 Learning path](README.md) · **37/49** · ◀ [Modern GPU & Model Architecture](./Modern_GPU_And_Model_Architecture.ipynb) · [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb) ▶

# Training Kernels & Where Training Memory Goes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Training_Kernels_And_Memory.ipynb)

Everything in [GPU Architecture & CUDA Kernels](./GPU_Architecture_And_CUDA_Kernels.ipynb) was
inference: one forward pass, no gradients, no optimizer, no second GPU. Training changes four
things, and each one has a kernel behind it:

| | the kernel | what is different |
|---|---|---|
| **the backward pass** | [`07_rmsnorm_backward.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/07_rmsnorm_backward.cu) | ~2x the FLOPs, and dW is a reduction *across the whole batch* |
| **the optimizer** | [`08_adamw.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/08_adamw.cu) | 0.4 FLOP/byte, touching every parameter every step |
| **low precision** | [`09_fp8_scaling.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/09_fp8_scaling.cu) | the scale factor becomes part of the algorithm |
| **many GPUs** | [`10_collectives.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/10_collectives.cu) | an all-reduce of every gradient, every step |

Two threads run through all four.

**The first is that training is a memory problem long before it is a compute problem.** A 7B
model in fp32 AdamW needs 112 GB of weights, gradients and optimizer state before a single
activation exists, on a card that has 80 GB. Every technique people reach for — mixed
precision, ZeRO, 8-bit optimizer states, LoRA, activation checkpointing — is about that
number, and none of them is about making the arithmetic faster.

**The second is reproducibility.** The backward pass sums a gradient contribution from every
row in the batch, and the obvious way to do that is an atomic add. It is correct. It is also
not reproducible: floating-point addition is not associative, so the result depends on the
order the blocks happened to finish in, and two runs of the same step on the same data give
weights that differ in the last bits. Over thousands of steps that compounds into a run you
cannot bisect. `07_rmsnorm_backward.cu` shows both versions, and the eval harness in this repo
**verifies the distinction mechanically** — by re-running each kernel under a shuffled block
order and comparing exact checksums.

No GPU needed: every kernel here compiles with `g++` against a CPU shim that runs one thread
per CUDA thread. With a GPU you also get the timings.

In [ ]:
# Setup. On Colab this clones the repo; run it and everything below works.
import os, subprocess, sys, math, json, uuid
from pathlib import Path

def find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "kernels" / "Makefile").exists():
            return cand
    return None

URL = "https://github.com/sugeerth/gpu-training-notebooks"
BRANCH = "claude/serving-optimization-notebooks-jyerty"   # until this lands on main

REPO = find_repo()
if REPO is None:
    dest = Path("/content/gpu-training-notebooks")
    if not (dest / "kernels" / "Makefile").exists():
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", URL, str(dest)], check=True)
        if not (dest / "kernels" / "Makefile").exists():
            subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=str(dest))
            subprocess.run(["git", "checkout", "FETCH_HEAD"], cwd=str(dest))
    REPO = dest
KERNELS = REPO / "kernels"

def sh(cmd, cwd=KERNELS, limit=7000):
    """Run a command and show its output, minus the ##KB## line the eval harness reads."""
    p = subprocess.run(cmd, shell=True, cwd=str(cwd), capture_output=True, text=True)
    clean = "\n".join(l for l in (p.stdout or "").splitlines() if not l.startswith("##KB##"))
    out = clean[-limit:]
    if out:
        print(out, end="" if out.endswith("\n") else "\n")
    if p.returncode != 0 and p.stderr:
        print((p.stderr or "")[-2000:], file=sys.stderr)
    return p.returncode

def peek(filename, symbol, before=14):
    """Show one function from the real source file, comment block included."""
    lines = (KERNELS / filename).read_text().split("\n")
    start = next((i for i, l in enumerate(lines) if symbol in l and ("(" in l or "=" in l)), None)
    if start is None:
        print(f"{symbol} not found in {filename}"); return
    top = start
    while top > 0 and (lines[top - 1].startswith("//") or lines[top - 1].strip() == ""
                       or lines[top - 1].startswith(("__device__", "__global__", "constexpr"))):
        top -= 1
        if start - top > before * 6:
            break
    end = start
    while end < len(lines) - 1 and lines[end] != "}":
        end += 1
    print("\n".join(lines[top:end + 1]))

print("repo :", REPO)
try:
    import torch
    HAVE_GPU = torch.cuda.is_available()
    print("gpu  :", torch.cuda.get_device_name(0) if HAVE_GPU else "none — correctness only")
except ImportError:
    torch, HAVE_GPU = None, False
    print("gpu  : no torch — correctness only")

## Part 1 · The backward pass, and the end of reproducibility

RMSNorm forward is `y = x·r·w` with `r = (mean(x²)+ε)^(-1/2)`. Differentiate it:

$$\frac{\partial L}{\partial W_i} = \sum_{\text{rows}} dy_i \, x_i \, r \qquad
\frac{\partial L}{\partial X_i} = r\left(dy_i w_i - x_i r^2 \frac{S}{H}\right), \quad
S = \sum_j dy_j w_j x_j$$

`dX` is elementwise given `S` — one row's business. **`dW` is a reduction over the entire
batch**, and that asymmetry is the whole story of the backward pass. Every row of every
sequence contributes to the same `H` floats, so the kernel has to combine thousands of partial
results, and *how* it combines them decides whether your training run reproduces.

In [ ]:
# The obvious implementation, and the one that costs you reproducibility.
peek("07_rmsnorm_backward.cu", "bwd_atomic")

`atomicAdd` is correct: every ordering produces a valid answer, and the spread between them is
at the level of fp32 rounding. It matters because a training run *compounds* it, and because
"bit-identical given the same inputs" is the property that makes a regression bisectable.

The fix is a fixed-shape two-stage reduction — each block accumulates privately, then a second
kernel sums the partials in index order. It costs `G × H` floats of scratch (4 MB here) and one
extra kernel launch. That is the entire price of reproducible training, and the usual reason
people do not have it is that nobody asked.

In [ ]:
sh("make --no-print-directory 07_rmsnorm_backward")

### Proving the difference rather than asserting it

The claim "variant 1 is order-dependent and variant 2 is not" is exactly the kind of thing that
is true when written and false a year later. So it is checked.

`kernelbench/shim/cuda_shim.hpp` runs blocks in an order it can **permute** — CUDA guarantees
nothing about block order, so the shim is entitled to shuffle it. Set `KB_SHIM_SHUFFLE=<seed>`,
run the same binary twice with different seeds, and compare exact checksums of the output. A
kernel whose answer depends on block order changes; one that does not, does not.

This reproduces GPU non-determinism **on a CPU, deterministically, in under a second** — a
class of bug that otherwise only appears as flakiness on real hardware.

In [ ]:
# Run the same binary under four different block orders and compare exact checksums.
import subprocess
BIN = KERNELS / "build" / "cpu" / "07_rmsnorm_backward"
if not BIN.exists():
    sh("make --no-print-directory HAVE_NVCC=no build/cpu/07_rmsnorm_backward")

def checksums(seed=None):
    env = dict(os.environ)
    if seed is not None:
        env["KB_SHIM_SHUFFLE"] = str(seed)
    out = subprocess.run([str(BIN)], capture_output=True, text=True, env=env).stdout
    line = next(l for l in out.splitlines() if l.startswith("##KB##"))
    return json.loads(line[7:])["variants"]

base = checksums()
names = [v["name"] for v in base]
rows = {n: [] for n in names}
for seed in (1, 5, 42, 1337):
    for v in checksums(seed):
        rows[v["name"]].append(v["checksum"][:10])

print(f"{'variant':<34} {'checksums under 4 shuffled block orders':<48} verdict")
print("-" * 104)
for n in names:
    uniq = len(set(rows[n]))
    verdict = "ORDER-DEPENDENT" if uniq > 1 else "reproducible"
    print(f"{n:<34} {' '.join(sorted(set(rows[n]))[:3]):<48} {verdict}")
print("""
The atomic variant gives a different answer depending on which block got there first.
The deterministic variants give the same bits every time, on every seed.

That is not a bug in either kernel — it is a property you either chose or did not. The eval
harness turns it into a gate: kernels/kernelbench.json declares which variants are *allowed*
to be order-dependent, and the check fails in both directions — an undeclared one is a
reproducibility hazard, and a declared one that turns out stable means the declaration is
stale.""")

## Part 2 · The optimizer step, and the memory budget

AdamW does about 11 FLOPs per parameter against 28 bytes of traffic — **0.4 FLOP/byte**, an
order of magnitude below even the memory-bound kernels in the inference notebook. It is pure
bandwidth, and the only thing that makes it faster is moving fewer bytes.

Which is exactly what a framework composing primitives does not do. `m.mul_(b1).add_(g)` and
its four siblings are five separate kernels, each making a full round trip to HBM for an
operation with no arithmetic to hide it: 48 bytes per parameter instead of 28.

In [ ]:
sh("make --no-print-directory 08_adamw")

The table that program prints at the end is the one worth keeping, and the cell below turns it
into the model this notebook uses for everything after it — and that the
[training planner](https://github.com/sugeerth/gpu-training-notebooks/blob/main/demo/training-planner.html)
on the demo site reproduces in your browser.

Two parts of it deserve a note before you use it.

**bf16 with an fp32 master copy does not save optimizer memory.** `p_bf16(2) + g_bf16(2) +
master(4) + m(4) + v(4)` is 16 bytes per parameter — the same as plain fp32 AdamW. What it buys
is halved *activation* memory, halved gradient *communication*, and tensor cores in the forward
and backward passes. The master copy exists because a typical update is ~1e-3 against a weight
of order 1, and in bf16's ~3 decimal digits `1.0 + 0.001` rounds straight back to `1.0` — the
update is silently discarded and the model stops learning while every metric looks fine.

**Activation memory is the term people forget**, and it is the one that scales with sequence
length and batch. The estimate below is the Megatron-LM formula: per layer, per token,
`s·b·h·(34 + 5·a·s/h)` bytes in 16-bit. The `5·a·s/h` term is the attention score matrix, and
it is quadratic in sequence length — which is why long-context training needs either
FlashAttention (no score matrix) or recomputation.

In [ ]:
# The training memory and communication model. This function is the source of truth: the
# demo site's planner reproduces it in JavaScript and tools/verify_console.py checks the two
# agree over thousands of configurations.
def training_plan(params_b=7.0, recipe="bf16+master", gpus=8, vram_gb=80.0,
                  seq=4096, micro_batch=1, layers=32, hidden=4096, heads=32,
                  zero_stage=1, checkpointing="none", flash_attention=True,
                  link_gbps=450.0, link_lat_us=3.0):
    """Memory per GPU and communication per step for one training configuration.

    Returns a dict of GB figures plus the all-reduce cost. Everything is per-GPU except
    `grad_allreduce_gb`, which is the payload each GPU contributes.
    """
    P = params_b * 1e9

    # Bytes per parameter, by recipe. `w` folds together every copy of the weights that has
    # to exist at once — for mixed precision that is the bf16 copy the forward pass reads
    # plus the fp32 master the optimizer updates.
    RECIPES = {
        "fp32":         dict(w=4.0, g=4.0, m=4.0, v=4.0, trainable=1.0),
        "bf16+master":  dict(w=6.0, g=2.0, m=4.0, v=4.0, trainable=1.0),
        "bf16+8bit":    dict(w=6.0, g=2.0, m=1.0, v=1.0, trainable=1.0),
        "lora-r16":     dict(w=2.0, g=2.0, m=4.0, v=4.0, trainable=0.001),
    }
    r = RECIPES[recipe]
    t = r["trainable"]

    # ZeRO shards the *replicated* state across data-parallel ranks. Stage 1 the optimizer
    # states, stage 2 also the gradients, stage 3 also the parameters themselves.
    shard_opt = gpus if zero_stage >= 1 else 1
    shard_grad = gpus if zero_stage >= 2 else 1
    shard_param = gpus if zero_stage >= 3 else 1

    weights_gb = P * r["w"] / shard_param / 1e9
    grads_gb = P * t * r["g"] / shard_grad / 1e9
    optim_gb = P * t * (r["m"] + r["v"]) / shard_opt / 1e9

    # Activations, from Megatron-LM's accounting. Two terms per layer, at 16-bit:
    #
    #   s*b*h*34        the ordinary activations — linear in sequence length
    #   5*a*s^2*b       the attention score matrix and friends — QUADRATIC in s
    #
    # The second term is what FlashAttention deletes: it never materializes the score matrix,
    # so there is nothing to store. At s=4096, a=32 it is 2.7 GB per layer against 0.6 GB for
    # everything else, so whether it is present decides the entire activation budget. Nobody
    # trains long context without it, which is why it defaults to on here.
    base = seq * micro_batch * hidden * 34.0
    attn = 5.0 * heads * seq * seq * micro_batch
    per_layer = base + (0.0 if flash_attention else attn)
    if checkpointing == "full":
        # Store only each layer's input and recompute the rest: 2*s*b*h bytes per layer,
        # bought with roughly one extra forward pass (~33% more compute).
        per_layer = 2.0 * seq * micro_batch * hidden
    elif checkpointing == "selective":
        # Megatron's selective recompute: drop the quadratic term, keep the rest. With
        # FlashAttention this is already where you are, which is why the two coincide.
        per_layer = base
    act_gb = layers * per_layer / 1e9

    total_gb = weights_gb + grads_gb + optim_gb + act_gb
    fits = total_gb <= vram_gb

    # Gradient all-reduce. ZeRO-2 and above already reduce-scatter the gradients as part of
    # the sharding, so the payload is the same 2(R-1)/R factor either way.
    payload = P * t * r["g"]
    eff = 2.0 * (gpus - 1) / gpus if gpus > 1 else 0.0
    ring_steps = 2 * (gpus - 1) if gpus > 1 else 0
    comm_bytes = payload * eff
    ring_s = comm_bytes / (link_gbps * 1e9) + ring_steps * link_lat_us * 1e-6
    direct_s = comm_bytes / (link_gbps * 1e9) + 2 * link_lat_us * 1e-6

    # Compute, at a stated 40% model FLOPs utilization — the number a well-tuned run reaches
    # and the one worth planning against, rather than peak.
    tokens = seq * micro_batch * gpus
    recompute = 4.0 / 3.0 if checkpointing == "full" else 1.0
    flops = 6.0 * P * tokens * recompute

    return dict(
        weights_gb=weights_gb, grads_gb=grads_gb, optim_gb=optim_gb, act_gb=act_gb,
        total_gb=total_gb, vram_gb=vram_gb, fits=fits,
        headroom_gb=vram_gb - total_gb,
        grad_allreduce_gb=payload / 1e9, comm_gb=comm_bytes / 1e9,
        ring_ms=ring_s * 1e3, direct_ms=direct_s * 1e3, ring_steps=ring_steps,
        tokens_per_step=tokens, step_flops=flops,
        binding=("memory" if not fits else "compute"),
    )


p = training_plan()
print("7B, bf16 + fp32 master, 8 GPUs, ZeRO-1, 4k sequence, micro-batch 1\n")
for k in ("weights_gb", "grads_gb", "optim_gb", "act_gb", "total_gb"):
    print(f"  {k:<14} {p[k]:8.2f} GB")
print(f"  {'headroom':<14} {p['headroom_gb']:8.2f} GB on an 80 GB card"
      f"   -> {'FITS' if p['fits'] else 'DOES NOT FIT'}")
print(f"\n  gradient payload   {p['grad_allreduce_gb']:6.2f} GB")
print(f"  moved per GPU      {p['comm_gb']:6.2f} GB  (the 2(R-1)/R factor)")
print(f"  ring   {p['ring_ms']:7.2f} ms  ({p['ring_steps']} serialized steps)")
print(f"  direct {p['direct_ms']:7.2f} ms  (2 steps, same bytes)")

In [ ]:
# The recipes, side by side, on the configuration above.
def show(**kw):
    base = dict(params_b=7.0, gpus=8, vram_gb=80.0, seq=4096, micro_batch=1,
                layers=32, hidden=4096, heads=32, zero_stage=1, checkpointing="none",
                flash_attention=True)
    base.update(kw)
    return training_plan(**base)

print(f"{'recipe':<16}{'zero':>5}{'ckpt':>11}{'flash':>7}{'weights':>9}{'grads':>8}"
      f"{'optim':>8}{'activ':>8}{'total':>9}  verdict")
print("-" * 95)
rows = [
    ("bf16+master", 0, "none", False),
    ("bf16+master", 0, "none", True),
    ("fp32",        0, "none", True),
    ("fp32",        1, "none", True),
    ("bf16+master", 1, "none", True),
    ("bf16+master", 2, "none", True),
    ("bf16+master", 3, "none", True),
    ("bf16+master", 1, "full", True),
    ("bf16+8bit",   1, "none", True),
    ("lora-r16",    0, "none", True),
]
for recipe, z, ck, fa in rows:
    r = show(recipe=recipe, zero_stage=z, checkpointing=ck, flash_attention=fa)
    print(f"{recipe:<16}{z:>5}{ck:>11}{('yes' if fa else 'NO'):>7}{r['weights_gb']:>9.1f}"
          f"{r['grads_gb']:>8.1f}{r['optim_gb']:>8.1f}{r['act_gb']:>8.1f}{r['total_gb']:>9.1f}  "
          f"{'fits' if r['fits'] else 'DOES NOT FIT'}")

print("""
Start with the first two rows, which differ only in FlashAttention. Without it the attention
score matrix alone is 86 GB — more than the entire rest of the model — because it is quadratic
in sequence length. That single row is why long-context training was impractical before 2022
and why every framework now defaults to it.

After that, read down the total column. Plain fp32 is 112 GB of state before activations, on
an 80 GB card. Nothing about that is a compute problem, and no amount of tuning the optimizer
kernel touches it.

Read across instead, and the levers separate cleanly:
  * FlashAttention       deletes the s^2 activation term outright. Free; do it first.
  * mixed precision      halves gradients and activations; optimizer state is unchanged
  * ZeRO 1 -> 2 -> 3     divides progressively more of the *replicated* state by the world size
  * checkpointing        trades ~33% more compute for most of what is left
  * 8-bit states         takes m and v from 8 bytes per parameter to 2
  * LoRA                 makes 99.9% of the parameters untrainable, so they need no state at all

They compose, and the order to reach for them is the order of what binds. That is what the
planner on the demo site is for.""")

## Part 3 · FP8, and the two ways a low-precision run silently stops learning

FP8 halves the bytes again and doubles tensor-core throughput on Hopper and Blackwell. It is
also the first format where **the scale factor is part of the algorithm**, because the dynamic
range is too small for an unscaled tensor to fit:

| | exponent / mantissa | max | used for |
|---|---|---|---|
| `e4m3` | 4 / 3 | ±448 | forward: activations and weights, which want the precision |
| `e5m2` | 5 / 2 | ±57344 | gradients, which span orders of magnitude and want the range |

Two failure modes, both silent, both invisible unless counted:

- **saturation** — the scale is too small, large values clamp at ±448, and the biggest
  activations' gradients are truncated. Loss keeps decreasing, slightly wrong.
- **underflow** — the scale is too large, small values round to zero, and a channel's gradient
  disappears entirely.

`09_fp8_scaling.cu` implements four scaling granularities and counts both.

In [ ]:
sh("make --no-print-directory 09_fp8_scaling")

The progression in the `L2 vs fp32` column is the point: per-tensor scaling is dominated by
whichever channel happens to hold the largest outlier, and transformer activations reliably
contain channels two orders of magnitude above the rest. Per-row and per-128-block scaling cost
one fp32 scale per group — about 3% overhead — and buy back most of that.

**Delayed scaling** is what production actually uses, and it is where the saturation counter
earns its keep. Computing an amax needs a full pass over the tensor *before* you can quantize
it, which is two passes on a kernel that exists to move fewer bytes. So Transformer Engine uses
the amax from previous steps. When the tensor's magnitude moves — a loss spike, a warmup
transition, a new sequence length — the stale scale is too small and values clip. Nothing
raises an error; the run just trains on clipped gradients.

## Part 4 · Many GPUs, and the communication tax

Data-parallel training ends every step by summing gradients across all ranks. For an 8B model
in bf16 that is 16 GB, all-reduced, on every step. Whether that costs 5% or 50% of the step is
decided by an algorithm.

- **naive** — every rank fetches everyone else's buffer: `R·N` bytes per rank, growing without
  bound in the number of ranks.
- **ring reduce-scatter + all-gather** — `2(R−1)/R · N`, which tends to `2N` however large the
  cluster gets. `2(R−1)` serialized steps.
- **direct** — the same bytes in 2 steps instead of `2(R−1)`.

Under the standard α–β cost model the direct version is therefore *never slower*, which raises
the question of why rings are the default. The answer is topology, not arithmetic: a ring
visits each link once per step and embeds on a hierarchical network without oversubscribing
anything, while all-to-all traffic congests the inter-node links the moment it leaves a node.
NCCL switches between them by message size.

In [ ]:
sh("make --no-print-directory 10_collectives")

In [ ]:
# What the interconnect costs you, for the configuration from Part 2.
LINKS = [("NVLink 4 (intra-node)", 450.0, 3.0), ("PCIe 5 x16", 55.0, 8.0),
         ("InfiniBand NDR 400G", 50.0, 5.0), ("100 GbE", 12.0, 30.0)]

print(f"{'interconnect':<24}{'all-reduce':>12}{'ring':>10}{'direct':>10}"
      f"{'% of a 400 ms step':>21}")
print("-" * 78)
for name, gbps, lat in LINKS:
    r = training_plan(params_b=8.0, gpus=8, link_gbps=gbps, link_lat_us=lat)
    print(f"{name:<24}{r['comm_gb']:>9.1f} GB{r['ring_ms']:>9.0f}ms{r['direct_ms']:>9.0f}ms"
          f"{100*r['ring_ms']/400.0:>20.1f}%")

print("""
That last column is why the parallelism plan is chosen by the interconnect and not by the
model. On NVLink the all-reduce disappears into the noise; on 100 GbE it is five times the
step. The three responses, in the order people reach for them:

  * overlap it   — bucket the gradients and start the all-reduce for early layers while the
                   backward pass is still running on later ones. Free, and always worth doing.
  * shrink it    — ZeRO already reduce-scatters, so the payload is the same; but sharding the
                   optimizer means each rank only all-gathers the parameters it needs.
  * avoid it     — tensor-parallelism inside a node where the links are fast, data-parallelism
                   across nodes where they are not.""")

## Part 5 · Running the whole thing as an eval

The four kernels above are not a demo — they are a suite, and the repo scores them on every
push with [`kernelbench`](https://github.com/sugeerth/gpu-training-notebooks/tree/main/kernelbench),
on a machine with no GPU.

Six dimensions, each of which fails independently: **build**, **correctness**,
**variants**, **determinism**, **mutation** and **efficiency**. The one to read first is
mutation — it is the only dimension that measures the *test* rather than the code. A suite
where every kernel passes and no injected bug is caught has established nothing.

In [ ]:
sh("python -m kernelbench eval kernels/ --only 07_rmsnorm_backward.cu --only 08_adamw.cu "
   "--only 09_fp8_scaling.cu --only 10_collectives.cu", cwd=REPO, limit=4000)

## What to take away

1. **The backward pass is a reduction across the batch**, and how you combine it decides
   whether the run reproduces. Atomics are correct and not reproducible; a fixed two-stage
   reduction costs one launch and some scratch.
2. **Training is a memory problem.** 16 bytes per parameter of state before activations, and
   activations that grow with `s·b·h` and — without FlashAttention or recompute — with `s²`.
3. **The optimizer step is 0.4 FLOP/byte.** Fuse it and stop; there is nothing else there.
4. **FP8 makes the scale part of the algorithm**, with two silent failure modes that only a
   counter will show you.
5. **The interconnect chooses the parallelism plan**, not the model.

### Next

- [Modern GPU & Model Architecture](./Modern_GPU_And_Model_Architecture.ipynb) — Hopper and
  Blackwell, and the model architectures built to exploit them: MLA and fine-grained MoE.
- [Measuring GPU Code Honestly](./Measuring_GPU_Code_Honestly.ipynb) — why the timings in this
  notebook are taken the way they are.
- [From Fine-Tune to Production](./From_FineTune_To_Production.ipynb) — what happens to a model
  after the training loop finishes.